# Supplementary figure: regression of scan structure

A single-subject worked example, for supplementary section *Regression of scan structure*
(`sfig:example_timeseries`).

The argument the figure has to carry, panel by panel:

| panel | shows | sentence it supports |
| --- | --- | --- |
| A | the HRF-convolved block design | "block designs alternate between conditions that can span tens of seconds" |
| B, C | all 400 parcels before and after task regression | what the design puts into, and regression takes out of, the data |
| D | the most task-driven region's timeseries | "any brain region whose activity tracks this design inherits this dependence" |
| E | ACFs before/after, against the design's **own** ACF | "the temporal structure of the task is itself autocorrelated" |
| F | INT per region before vs after | "this inherited structure affects the INT estimate directly" |

**Unfiltered.** Timeseries are detrended and nuisance-regressed but *not* bandpassed, matching
how the HCP-YA results in the main text were produced (`timescale_auc_zero_crossing`, no `bp`
token). It also matters for the argument: a 0.01 Hz highpass would itself attenuate part of the
block-design power this figure exists to show.

The notebook re-derives the example subject's scan from scratch rather than reading a cached
INT, and asserts that the result reproduces the cached value.

In [ ]:
import sys, os, warnings, subprocess
subprocess.run(["pip", "install", ".", "--quiet"], check=True,
               cwd="/mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales")
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

%matplotlib inline

sys.path.extend([r'/mnt/storage_ssd_raid/research_projects/snaplab_tools'])

from src.data import DataLoader
from src.timescales import Timescales
# INT estimation comes from snaplab_tools; src.timescales owns only the caching layer.
from snaplab_tools.timescales import compute_acf, TIMESCALE_METHODS
from src.supporting_data import load_supporting_data
from src.plotting import plot_task_regression_example, figure_scratch_dir, setup_notebook_style
from snaplab_tools.plotting.utils import save_panel

setup_notebook_style()


In [ ]:
dataset      = 'HCP-YA'
task_scan    = 'tfMRIWMLR'   # working memory: blocked, 8 x 27.5 s blocks, TR 0.72 s
n_regions    = 400
seg          = 'Schaefer4007'
space        = 'fsLR'
session      = '01'
modality     = 'MRI3T'

# The INT estimator used throughout the main text.
int_method   = 'auc_zero_crossing'
output_type  = f'timescale_{int_method}'

# Set to a subject ID (e.g. '214423') to pin the example; None selects the median
# subject in the next cell.
example_subject = None

## Choose the example subject

A "single subject" figure invites the question of whether the subject was picked to flatter the
argument. So the choice is made from the already-cached group results and stated: the subject
whose INT reduction is closest to the **median** across everyone with this scan. The cell prints
the group distribution, so the caption can report where the example sits in it.

In [ ]:
selector = Timescales(dataset=dataset, timeseries_data=None, load_behavior=False, verbose=False)
selector.load_scan_registry(f'scan_registry_{dataset}.csv')

# Restrict to this task before loading, so we read ~1k cached files rather than ~15k.
selector.scan_registry = [s for s in selector.scan_registry if s.task == task_scan]

selector.get_precomputed_outputs(output_type=output_type)
cached_before = dict(selector.data_dict)
selector.get_precomputed_outputs(output_type=f'{output_type}_resids')
cached_after = dict(selector.data_dict)

rows = []
for scan_key, before in cached_before.items():
    after = cached_after.get(scan_key)
    if before is None or after is None:
        continue
    before, after = np.asarray(before, float), np.asarray(after, float)
    ok = np.isfinite(before) & np.isfinite(after)
    if ok.sum() < n_regions // 2:   # a scan where most regions have no zero crossing
        continue
    rows.append({
        'scan_key': scan_key,
        'sub':      selector._parse_scan_string(scan_key)['sub'],
        'int_before': np.nanmean(before[ok]),
        'int_after':  np.nanmean(after[ok]),
    })

df_group = pd.DataFrame(rows)
df_group['delta'] = df_group['int_after'] - df_group['int_before']

print(f'{len(df_group)} usable {task_scan} scans')
print(f"  INT before: {df_group['int_before'].mean():.3f} +/- {df_group['int_before'].std():.3f} s")
print(f"  INT after:  {df_group['int_after'].mean():.3f} +/- {df_group['int_after'].std():.3f} s")
print(f"  change:     median {df_group['delta'].median():.3f} s, "
      f"{100 * (df_group['delta'] < 0).mean():.1f}% of scans reduced")

In [ ]:
if example_subject is None:
    median_delta = df_group['delta'].median()
    pick = df_group.loc[(df_group['delta'] - median_delta).abs().idxmin()]
else:
    pick = df_group.loc[df_group['sub'] == str(example_subject)].iloc[0]

subject_id = pick['sub']
pctile = 100.0 * (df_group['delta'] < pick['delta']).mean()

print(f'example subject: {subject_id}')
print(f"  INT before {pick['int_before']:.3f} s, after {pick['int_after']:.3f} s "
      f"(change {pick['delta']:+.3f} s)")
print(f'  sits at the {pctile:.1f}th percentile of the group change distribution')

## Rebuild that subject's scan

`frequency_filter=False` is what [timescales_process.ipynb](timescales_process.ipynb) uses, so
this reproduces the pipeline exactly rather than approximating it — the assertion two cells down
checks that.

In [ ]:
loader = DataLoader(dataset=dataset, load_behavior=False, verbose=False)
loader.register_scans(subject_ids=[subject_id], sessions=[session], tasks=[task_scan],
                      space=space, seg=seg, modality=modality)

timeseries_data = loader.load_batch(detrend=True, nuisance_regression=True,
                                    frequency_filter=False, verbose=True)

scan_key = list(timeseries_data)[0]
tr = float(loader.get_tr(modality))
print(f'{scan_key}\n  shape {timeseries_data[scan_key].shape}, TR {tr} s')

In [ ]:
processor = Timescales(dataset=dataset, timeseries_data=timeseries_data,
                       load_behavior=False, verbose=False)
processor.regress_task_structure()

raw_data      = processor.timeseries_data[scan_key]
residual_data = processor.timeseries_residuals[scan_key]
design_matrix = processor.design_matrices[scan_key]
task_events   = processor.task_events[scan_key]
acf_design    = processor.task_events_acf[scan_key]   # ACF of the flattened task regressor

if residual_data is None:
    raise RuntimeError(f'task regression failed for {scan_key}')

task_cols = [c for c in design_matrix.columns
             if not c.startswith('drift_') and c != 'constant']
print(f'{len(task_cols)} task regressors: {task_cols}')
print(f'{len(task_events)} blocks, {task_events["duration"].iloc[0]:.1f} s each')
display(task_events)

In [ ]:
acf_before = compute_acf(raw_data, use_fft=True)
acf_after  = compute_acf(residual_data, use_fft=True)

int_fn = TIMESCALE_METHODS[int_method]
int_before = int_fn(acf_before, tr)
int_after  = int_fn(acf_after, tr)

print(f'INT before {np.nanmean(int_before):.3f} s, after {np.nanmean(int_after):.3f} s')

# The whole figure rests on this scan being the same one the group analysis used. If the
# recomputation drifts from the cached value, something upstream changed and the figure is
# no longer illustrating the published pipeline.
np.testing.assert_allclose(np.nanmean(int_before), pick['int_before'], rtol=1e-6)
np.testing.assert_allclose(np.nanmean(int_after),  pick['int_after'],  rtol=1e-6)
print('reproduces the cached group values')

In [ ]:
supporting_data = load_supporting_data(
    os.path.join(loader.output_dir, 'data', 'supporting_data'), n_regions=n_regions)
sa_axis = supporting_data['df_neuro']['SA_Axis'].values
print('S-A axis:', sa_axis.shape, f'ranks {sa_axis.min():.0f}-{sa_axis.max():.0f}')

## Draw the figure

In [ ]:
fig, axes, stats = plot_task_regression_example(
    raw_data=raw_data,
    residual_data=residual_data,
    design_matrix=design_matrix,
    acf_raw=acf_before,
    acf_resid=acf_after,
    acf_design=acf_design,
    int_raw=int_before,
    int_resid=int_after,
    tr=tr,
    task_events=task_events,
    brain_map=sa_axis,
    max_lag_seconds=20.0,
)

nas_scratch_dir = figure_scratch_dir(dataset)
# 400 imshow rows x 405 columns, twice -- worth more than the 300 dpi a line panel needs.
print(save_panel(fig, nas_scratch_dir,
                 f'sfig_task_regression_{dataset}_sub-{subject_id}_{task_scan}', dpi=400))
plt.show()

## Numbers for the caption

In [ ]:
print(f'Subject {subject_id}, {task_scan}, TR {tr} s, {raw_data.shape[1]} frames, '
      f'{n_regions} parcels.')
print(f'Selected as the median of {len(df_group)} scans by INT change '
      f'({pctile:.0f}th percentile).')
print()
print(f"Design: {len(task_events)} blocks of {task_events['duration'].iloc[0]:.1f} s, "
      f'{len(task_cols)} regressors.')
print(f"The design's own ACF stays positive out to "
      f"{stats['zero_crossing_design']:.1f} s.")
print()
print(f"Panel D region {stats['example_region'] + 1}: "
      f"{100 * stats['example_var_explained']:.0f}% of its variance explained by the design.")
print(f"Region-mean ACF crosses zero at {stats['zero_crossing_raw']:.1f} s before "
      f"regression and {stats['zero_crossing_resid']:.1f} s after.")
print(f"INT: {stats['int_raw_mean']:.2f} s -> {stats['int_resid_mean']:.2f} s "
      f"(median change {stats['int_median_change']:+.2f} s; "
      f"{100 * stats['frac_regions_reduced']:.0f}% of {stats['n_valid']} regions reduced).")
print()
print(f"Group context: INT {df_group['int_before'].mean():.2f} s -> "
      f"{df_group['int_after'].mean():.2f} s, "
      f"{100 * (df_group['delta'] < 0).mean():.0f}% of {len(df_group)} scans reduced.")